In [647]:
#!python -m venv venv

In [648]:
#!venv\Scripts\activate

!ollama pull phi3:mini
!ollama pull qwen2.5-coder:7b

In [649]:
#!ollama pull deepseek-llm

In [650]:
#!ollama pull deepseek-coder:6.7b

In [651]:
#!ollama pull gemma2:2b

In [652]:
#!ollama pull mistral:7b

In [653]:
#!ollama list

installer dépendences

In [654]:
#%pip install pandas sqlalchemy pymysql requests sentence-transformers faiss-cpu sqlglot

In [655]:
#%pip install sqlglot

In [ ]:
#%pip install flask flask-cors

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [656]:
import requests
import pandas as pd
import numpy as np
import faiss
import re
import time

from sentence_transformers import SentenceTransformer

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

import pymysql
print("PyMySQL installed")

change le chemain après: (normal terminal)
    cd C:\Users\Nouhaila\Desktop\test_db-master
    C:\xampp\mysql\bin\mysql.exe -u root employee
    source employees.sql;
FOR TEST : 
    SHOW TABLES;
    SELECT COUNT(*) FROM employees;

Connecté Mysql

from sqlalchemy import create_engine
import pandas as pd

DATABASE_URL = (
    "mysql+pymysql://root:@localhost:3306/employees"
)

engine = create_engine(DATABASE_URL)

print("MySQL connected")

Auto schéma extracteur

shema text

Database TEST

In [ ]:
from ai_init import *

Loading embedding model...


README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded
MySQL connected
Tables loaded
Schema metadata loaded
Relationships loaded
Table semantics generated
Business rules generated
Schema texts generated
Embeddings generated
FAISS ready
Initialization completed


: 

##########################################################
##########################################################
##########################################################
##########################################################

In [4]:
user_question = "Show departments that never had a manager salary above 100000"

In [661]:
tables_query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = DATABASE();
"""

tables_df = pd.read_sql(
    tables_query,
    engine
)

print(tables_df)

             table_name
0      current_dept_emp
1           departments
2              dept_emp
3  dept_emp_latest_date
4          dept_manager
5             employees
6              salaries
7                titles


In [662]:
schema_metadata = []

for table in tables_df["table_name"]:

    columns_query = f"""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
    AND table_name = '{table}';
    """

    columns_df = pd.read_sql(
        columns_query,
        engine
    )

    schema_metadata.append({
        "table": table,
        "columns": columns_df[
            "column_name"
        ].tolist()
    })

print(schema_metadata)

[{'table': 'current_dept_emp', 'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']}, {'table': 'departments', 'columns': ['dept_no', 'dept_name']}, {'table': 'dept_emp', 'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']}, {'table': 'dept_emp_latest_date', 'columns': ['emp_no', 'from_date', 'to_date']}, {'table': 'dept_manager', 'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']}, {'table': 'employees', 'columns': ['emp_no', 'birth_date', 'first_name', 'last_name', 'gender', 'hire_date']}, {'table': 'salaries', 'columns': ['emp_no', 'salary', 'from_date', 'to_date']}, {'table': 'titles', 'columns': ['emp_no', 'title', 'from_date', 'to_date']}]


In [663]:
relationships_query = """
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE REFERENCED_TABLE_NAME IS NOT NULL
AND TABLE_SCHEMA = DATABASE();
"""

relationships_df = pd.read_sql(
    relationships_query,
    engine
)

print(relationships_df)

     TABLE_NAME COLUMN_NAME REFERENCED_TABLE_NAME REFERENCED_COLUMN_NAME
0      dept_emp      emp_no             employees                 emp_no
1      dept_emp     dept_no           departments                dept_no
2  dept_manager      emp_no             employees                 emp_no
3  dept_manager     dept_no           departments                dept_no
4      salaries      emp_no             employees                 emp_no
5        titles      emp_no             employees                 emp_no


In [664]:
table_semantics = []

for item in schema_metadata:

    table_name = item["table"]

    columns = item["columns"]

    semantic_prompt = f"""
You are a database analyst.

Understand the purpose of this SQL table.

Table:
{table_name}

Columns:
{', '.join(columns)}

Explain:
- what this table represents
- what records it stores
- when this table should be used in SQL queries

Keep answer short.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral:7b",
            "prompt": semantic_prompt,
            "stream": False,
            "temperature": 0,
            "num_predict": 40
        }
    )

    semantic_text = response.json()[
        "response"
    ].strip()

    table_semantics.append({
        "table": table_name,
        "semantic": semantic_text
    })

print(table_semantics)

[{'table': 'current_dept_emp', 'semantic': 'This `current_dept_emp` table represents the current department assignments for employees at a given point in time. It stores records of employee IDs (`emp_no`), department numbers (`dept_no`), and the start and end dates of these assignments (`from_date` and `to_date` respectively). This table should be used in SQL queries when analyzing current or historical department assignments for employees.'}, {'table': 'departments', 'semantic': 'This table `departments` represents the structure of different departments within an organization. It stores records for each department with unique identifiers (`dept_no`) and their corresponding names (`dept_name`). This table should be used in SQL queries to retrieve or manipulate data related to department information, such as querying all departments or finding specific department details.'}, {'table': 'dept_emp', 'semantic': "This `dept_emp` table represents the employment details of employees within de

In [665]:
business_rules = []

for item in schema_metadata:

    table_name = item["table"]

    columns = item["columns"]

    if (
        "from_date" in columns
        and "to_date" in columns
    ):

        sample_query = f"""
        SELECT DISTINCT to_date
        FROM {table_name}
        ORDER BY to_date DESC
        LIMIT 5;
        """

        try:

            sample_df = pd.read_sql(
                sample_query,
                engine
            )

            values = sample_df[
                "to_date"
            ].astype(str).tolist()

            if "9999-01-01" in values:

                rule = f"""
In table {table_name},
to_date = '9999-01-01'
means current active record.
"""

                business_rules.append(
                    rule
                )

        except:
            pass

print(business_rules)

["\nIn table current_dept_emp,\nto_date = '9999-01-01'\nmeans current active record.\n", "\nIn table dept_emp,\nto_date = '9999-01-01'\nmeans current active record.\n", "\nIn table dept_emp_latest_date,\nto_date = '9999-01-01'\nmeans current active record.\n", "\nIn table dept_manager,\nto_date = '9999-01-01'\nmeans current active record.\n", "\nIn table salaries,\nto_date = '9999-01-01'\nmeans current active record.\n", "\nIn table titles,\nto_date = '9999-01-01'\nmeans current active record.\n"]


In [666]:
schema_texts = []

for item in schema_metadata:

    table_name = item["table"]

    columns = item["columns"]

    semantic_description = ""

    for semantic in table_semantics:

        if semantic["table"] == table_name:

            semantic_description = semantic[
                "semantic"
            ]

    text = f"""
Table: {table_name}

Meaning:
{semantic_description}

Columns:
{chr(10).join('- ' + c for c in columns)}
"""

    schema_texts.append(text)

print(schema_texts[0])


Table: current_dept_emp

Meaning:
This `current_dept_emp` table represents the current department assignments for employees at a given point in time. It stores records of employee IDs (`emp_no`), department numbers (`dept_no`), and the start and end dates of these assignments (`from_date` and `to_date` respectively). This table should be used in SQL queries when analyzing current or historical department assignments for employees.

Columns:
- emp_no
- dept_no
- from_date
- to_date



In [667]:
schema_embeddings = embedding_model.encode(
    schema_texts
).astype("float32")

print(schema_embeddings.shape)

(8, 384)


In [668]:
dimension = schema_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(schema_embeddings)

print("FAISS ready")

FAISS ready


In [669]:
query_embedding = embedding_model.encode(
    [user_question]
).astype("float32")

D, I = index.search(
    query_embedding,
    k=3
)

pre_normalization_schema = ""

for idx in I[0]:

    pre_normalization_schema += (
        schema_texts[idx]
    )

    pre_normalization_schema += "\n"

print(pre_normalization_schema)


Table: salaries

Meaning:
This `salaries` table represents employee payroll data with details such as employee ID (emp_no), salary amount, employment start date (from_date), and employment end date (to_date). It stores records of an employee's salary history throughout their tenure in the company. This table should be used in SQL queries for analyzing salary trends, calculating payroll totals, or generating reports on employee salaries over a specific period.

Columns:
- emp_no
- salary
- from_date
- to_date


Table: dept_manager

Meaning:
This `dept_manager` table represents the relationship between employees and departments, specifically those serving as department managers. It stores records of the employee ID (emp_no), department number (dept_no), the start date (from_date), and end date (to_date) for each manager in the respective department.

This table should be used in SQL queries when you want to analyze or retrieve information related to department managers, such as determin

In [670]:
important_tables = []

question_lower = user_question.lower()

for item in schema_metadata:

    table_name = item["table"]

    columns = item["columns"]

    combined_text = (
        table_name + " " +
        " ".join(columns)
    ).lower()

    keyword_score = 0

    for word in question_lower.split():

        if word in combined_text:

            keyword_score += 1

    if keyword_score > 0:

        important_tables.append({
            "table": table_name,
            "score": keyword_score
        })

important_tables = sorted(
    important_tables,
    key=lambda x: x["score"],
    reverse=True
)

print(important_tables)

[{'table': 'departments', 'score': 2}, {'table': 'dept_manager', 'score': 2}, {'table': 'salaries', 'score': 2}, {'table': 'current_dept_emp', 'score': 1}, {'table': 'dept_emp', 'score': 1}, {'table': 'dept_emp_latest_date', 'score': 1}, {'table': 'employees', 'score': 1}, {'table': 'titles', 'score': 1}]


In [671]:
relationship_text = ""

for _, row in relationships_df.iterrows():

    relationship_text += f"""
{row['TABLE_NAME']}.{row['COLUMN_NAME']}
references
{row['REFERENCED_TABLE_NAME']}.{row['REFERENCED_COLUMN_NAME']}
"""

print(relationship_text)


dept_emp.emp_no
references
employees.emp_no

dept_emp.dept_no
references
departments.dept_no

dept_manager.emp_no
references
employees.emp_no

dept_manager.dept_no
references
departments.dept_no

salaries.emp_no
references
employees.emp_no

titles.emp_no
references
employees.emp_no



In [672]:
intent_prompt = f"""
You are a semantic database request clarifier.

Your task:
Clarify the user's request while preserving the EXACT original intent, business meaning, and domain terminology.

DATABASE CONTEXT:
{pre_normalization_schema}

DATABASE RELATIONSHIPS:
{relationship_text}

STRICT RULES:
- Rewrite ONLY in natural language
- NEVER generate SQL
- NEVER explain
- NEVER answer
- Preserve ALL original meaning
- Preserve ALL business meaning
- Preserve ALL logical meaning
- Preserve ALL temporal meaning
- Preserve ALL hierarchical meaning
- Preserve ALL causal meaning
- Preserve ALL aggregation meaning
- Preserve ALL historical meaning
- Preserve ALL comparison meaning
- Preserve ALL change/evolution meaning
- Preserve implied semantics
- Never weaken the meaning
- Never simplify business concepts
- Do not paraphrase important business concepts
- Preserve original domain terminology whenever possible
- Preserve ontology meaning exactly
- Preserve implicit business intent
- If a term has important business meaning, keep the original wording
- Use database vocabulary only when relevant
- Keep the request clear and short
- If the request is already clear, keep it nearly unchanged
- Prefer preserving the original wording
- Modify only ambiguous or unclear parts
- Do not replace domain-specific terminology
- Do not introduce synonyms for important concepts

IMPORTANT:
- average must stay average
- count must stay count
- max must stay max
- min must stay min
- promoted must stay promoted
- demoted must stay demoted
- increased must preserve chronological increase meaning
- decreased must preserve chronological decrease meaning
- changed over time must preserve historical evolution meaning
- never changed must preserve exact historical meaning
- current must preserve active/current-state meaning

User Request:
{user_question}

Clarified Request:
"""

In [673]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "mistral:7b",
        "prompt": intent_prompt,
        "stream": False,
        "temperature": 0,
        "num_predict": 30
    }
)

normalized_question = response.json()[
    "response"
].strip()

print(normalized_question)

Display the departments where no department manager has ever earned a salary exceeding one hundred thousand dollars.


In [674]:
query_embedding = embedding_model.encode(
    [normalized_question]
).astype("float32")

D, I = index.search(
    query_embedding,
    k=3
)

In [675]:
relevant_semantics = []

for idx in I[0]:

    relevant_semantics.append(
        table_semantics[idx]["semantic"]
    )

semantic_context = "\n".join(
    relevant_semantics
)

print(semantic_context)

This `dept_manager` table represents the relationship between employees and departments, specifically those serving as department managers. It stores records of the employee ID (emp_no), department number (dept_no), the start date (from_date), and end date (to_date) for each manager in the respective department.

This table should be used in SQL queries when you want to analyze or retrieve information related to department managers, such as determining tenure of a department manager, identifying current or past managers for a specific department, or finding out how many employees have served as managers across all departments over time.
This `dept_emp` table represents the employment details of employees within departments in a company. It stores records for each employee assignment, including their unique employee number (`emp_no`), department number (`dept_no`), start date (`from_date`), and end date (`to_date` or null if still employed).

This table should be used in SQL queries rel

In [676]:
expanded_indices = set(I[0])

retrieved_tables = []

for idx in I[0]:

    table_name = schema_metadata[idx]["table"]

    retrieved_tables.append(
        table_name
    )

for _, row in relationships_df.iterrows():

    source_table = row[
        "TABLE_NAME"
    ]

    target_table = row[
        "REFERENCED_TABLE_NAME"
    ]

    if source_table in retrieved_tables:

        for i, item in enumerate(
            schema_metadata
        ):

            if item["table"] == target_table:

                expanded_indices.add(i)

    if target_table in retrieved_tables:

        for i, item in enumerate(
            schema_metadata
        ):

            if item["table"] == source_table:

                expanded_indices.add(i)

print(expanded_indices)

{1, 2, 4, 5}


In [677]:
question_lower = user_question.lower()

for idx, item in enumerate(schema_metadata):

    table_name = item["table"]

    columns = item["columns"]

    combined_text = (
        table_name + " " +
        " ".join(columns)
    ).lower()

    for word in question_lower.split():

        if word in combined_text:

            expanded_indices.add(idx)

            break

In [678]:
mini_schema = ""

for idx in expanded_indices:

    mini_schema += schema_texts[idx]

    mini_schema += "\n"

print(mini_schema)


Table: current_dept_emp

Meaning:
This `current_dept_emp` table represents the current department assignments for employees at a given point in time. It stores records of employee IDs (`emp_no`), department numbers (`dept_no`), and the start and end dates of these assignments (`from_date` and `to_date` respectively). This table should be used in SQL queries when analyzing current or historical department assignments for employees.

Columns:
- emp_no
- dept_no
- from_date
- to_date


Table: departments

Meaning:
This table `departments` represents the structure of different departments within an organization. It stores records for each department with unique identifiers (`dept_no`) and their corresponding names (`dept_name`). This table should be used in SQL queries to retrieve or manipulate data related to department information, such as querying all departments or finding specific department details.

Columns:
- dept_no
- dept_name


Table: dept_emp

Meaning:
This `dept_emp` table re

In [679]:
sql_prompt = f"""
You are an expert MySQL SQL generator.

Your task:
Generate ONE valid MySQL query from the user request.

DATABASE SCHEMA:
{mini_schema}

DATABASE RELATIONSHIPS:
{relationship_text}

DATABASE SEMANTICS:
{semantic_context}

DATABASE BUSINESS RULES:
{chr(10).join(business_rules)}

SQL REASONING GUIDELINES:

- If the request asks about changes over time:
analyze historical records for the same entity.

- If the request asks whether something changed:
compare multiple historical rows for the same entity.

- If the request asks about increases or decreases:
compare earlier values with later values chronologically.

- If the request asks about current information:
prefer current active records using business rules.

- If the request asks about averages:
use aggregation functions.

- If the request asks about entities belonging to multiple groups:
analyze DISTINCT grouped relationships.

- If the request asks about history:
use historical tables instead of current snapshot tables.

- If the request asks about trends over time:
analyze sequences ordered by dates.

- If the request asks whether something evolved:
compare old and new values for the same entity.

- If the request asks about repeated changes:
count distinct historical values.

- If the request contains "never":
analyze all historical records, not only current records.

- If the request contains "ever":
analyze the complete historical timeline.

- If the request asks about historical events:
do not restrict analysis to current active rows unless explicitly requested.

- If the request contains negation:
carefully analyze the full historical scope before excluding entities.

- Distinguish carefully between:
  - current state
  - historical state
  - entire history

- If the request refers to all time:
avoid filtering only current records.

- If the request asks whether an entity NEVER had a condition:
exclude entities if the condition occurred at least once in history.

- Apply negation conditions to the complete entity history unless the request specifies current state only.

- Distinguish carefully between:
  - any related record
  - all related records
  - current related records
  - historical related records

- If one historical occurrence violates the condition:
the entity should be excluded when using NEVER logic.

- For department-level conditions:
analyze all related department records before filtering.

IMPORTANT RULES:
- Return ONLY SQL
- No markdown
- No explanation
- Use ONLY existing tables
- Use ONLY existing columns
- Never invent columns
- Never invent tables
- Use valid MySQL syntax
- Keep query simple
- Use LIMIT 10 unless aggregation is required

Question:
{normalized_question}

SQL:
"""

In [680]:
start = time.time()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "deepseek-coder:6.7b",
        "prompt": sql_prompt,
        "stream": False,
        "temperature": 0,
        "num_predict": 150
    }
)

end = time.time()

sql_query = response.json()[
    "response"
]

print(f"Generation time: {end - start:.2f} sec")

Generation time: 417.27 sec


In [681]:
sql_query = re.sub(
    r"```sql",
    "",
    sql_query
)

sql_query = re.sub(
    r"```",
    "",
    sql_query
)

sql_query = re.sub(
    r"<\\|.*?\\|>",
    "",
    sql_query
)

sql_query = re.sub(
    r"▁",
    "",
    sql_query
)

sql_query = re.sub(
    r"\s+",
    " ",
    sql_query
)

sql_query = sql_query.strip()

sql_query = sql_query.split(";")[0] + ";"

print(sql_query)

SELECT DISTINCT d.dept_name FROM departments d JOIN dept_manager dm ON d.dept_no = dm.dept_no AND dm.to_date = '9999-01-01' LEFT JOIN salaries s ON dm.emp_no = s.emp_no AND s.salary 100000 AND s.to_date = '9999-01-01' WHERE s.emp_no IS NULL;


In [682]:
import sqlglot

try:

    parsed = sqlglot.parse_one(
        sql_query,
        read="mysql"
    )

    print("SQL syntax valid")

except Exception as e:

    print("SQL syntax error")

    print(e)

SQL syntax error
Invalid expression / Unexpected token. Line 1, Col: 188.
   dm.dept_no AND dm.to_date = '9999-01-01' LEFT JOIN salaries s ON dm.emp_no = s.emp_no AND s.salary 100000 AND s.to_date = '9999-01-01' WHERE s.emp_no IS NULL;


In [683]:
try:

    test_query = f"""
    EXPLAIN {sql_query}
    """

    pd.read_sql(
        test_query,
        engine
    )

    print("SQL execution valid")

except Exception as e:

    print("SQL execution error")

    print(e)

SQL execution error
(pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MariaDB server version for the right syntax to use near '100000 AND s.to_date = '9999-01-01' WHERE s.emp_no IS NULL' at line 1")
[SQL: 
    EXPLAIN SELECT DISTINCT d.dept_name FROM departments d JOIN dept_manager dm ON d.dept_no = dm.dept_no AND dm.to_date = '9999-01-01' LEFT JOIN salaries s ON dm.emp_no = s.emp_no AND s.salary 100000 AND s.to_date = '9999-01-01' WHERE s.emp_no IS NULL;
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [684]:
clean_sql_for_repair = re.sub(
    r"<\|.*?\|>",
    "",
    sql_query
)

clean_sql_for_repair = re.sub(
    r"▁",
    "",
    clean_sql_for_repair
)

clean_sql_for_repair = re.sub(
    r"\s+",
    " ",
    clean_sql_for_repair
)

clean_sql_for_repair = clean_sql_for_repair.strip()

print(clean_sql_for_repair)

SELECT DISTINCT d.dept_name FROM departments d JOIN dept_manager dm ON d.dept_no = dm.dept_no AND dm.to_date = '9999-01-01' LEFT JOIN salaries s ON dm.emp_no = s.emp_no AND s.salary 100000 AND s.to_date = '9999-01-01' WHERE s.emp_no IS NULL;


In [685]:
sql_error = str(globals().get("e", ""))

repair_prompt = f"""
You are an expert MySQL SQL fixer.

Your task:
Repair the invalid SQL query.

DATABASE SCHEMA:
{mini_schema}

DATABASE RELATIONSHIPS:
{relationship_text}

ORIGINAL QUESTION:
{normalized_question}

INVALID SQL:
{clean_sql_for_repair}

DATABASE ERROR:
{sql_error}

RULES:
- Return ONLY corrected SQL
- No markdown
- No explanation
- Keep original meaning
- Use ONLY existing tables
- Use ONLY existing columns
- Fix syntax and logic errors

CORRECTED SQL:
"""

In [686]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "deepseek-coder:6.7b",
        "prompt": repair_prompt,
        "stream": False,
        "temperature": 0,
        "num_predict": 150
    }
)

repaired_sql = response.json()[
    "response"
].strip()

print(repaired_sql)

SELECT DISTINCT d.dept_name 
FROM departments d 
JOIN dept_manager dm ON d.dept_no = dm.dept_no AND dm.to_date = '9999-01-01' 
LEFT JOIN salaries s ON dm.emp_no = s.emp_no AND s.salary > 100000 AND s.to_date = '9999-01-01' 
WHERE s.emp_no IS NULL;


In [687]:
repaired_sql = re.sub(
    r"```sql",
    "",
    repaired_sql
)

repaired_sql = re.sub(
    r"```",
    "",
    repaired_sql
)

repaired_sql = repaired_sql.strip()

repaired_sql = repaired_sql.split(";")[0] + ";"

print(repaired_sql)

SELECT DISTINCT d.dept_name 
FROM departments d 
JOIN dept_manager dm ON d.dept_no = dm.dept_no AND dm.to_date = '9999-01-01' 
LEFT JOIN salaries s ON dm.emp_no = s.emp_no AND s.salary > 100000 AND s.to_date = '9999-01-01' 
WHERE s.emp_no IS NULL;


In [688]:
try:

    pd.read_sql(
        f"EXPLAIN {repaired_sql}",
        engine
    )

    print("Repaired SQL valid")

except Exception as e:

    print(e)

Repaired SQL valid


In [689]:
start = time.time()

try:

    df = pd.read_sql(
        repaired_sql,
        engine
    )

    end = time.time()

    print(df.head())

    print(
        f"Execution time: {end - start:.2f} sec"
    )

except Exception as e:

    print(e)

          dept_name
0  Customer Service
1       Development
2           Finance
3   Human Resources
4        Production
Execution time: 0.02 sec


In [ ]:
if len(df) == 0:

    result_preview = "No rows returned"

else:

    result_preview = df.head(10).to_string(
        index=False
    )

print(result_preview)

In [ ]:
human_prompt = f"""
You are a business data analyst.

USER QUESTION:
{user_question}

SQL RESULT:
{result_preview}

TASK:
Explain the result to a non-technical user.

RULES:
- Never generate SQL
- Never mention tables
- Never mention columns unless useful
- Never mention databases
- Use simple natural language
- Be concise
- If no records were found, clearly say so
- Base the answer ONLY on the provided result

ANSWER:
"""

In [ ]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "mistral:7b",
        "prompt": human_prompt,
        "stream": False,
        "temperature": 0.2,
        "num_predict": 120
    }
)

final_answer = response.json()[
    "response"
].strip()

print(final_answer)

In [ ]:
final_result = {
    "question": user_question,
    "normalized_question": normalized_question,
    "sql": repaired_sql,
    "answer": final_answer
}

print(final_result)

In [13]:
rows = df.to_dict(
    orient="records"
)

print(rows[:3])

NameError: name 'df' is not defined

# PIPELINE FUNCTIONS

In [5]:
def run_pipeline(user_question):

    print("Pipeline function not implemented yet")

    return {
        "question": user_question
    }

In [6]:
result = run_pipeline(
    "Show employees with above-average salaries"
)

print(result)

Pipeline function not implemented yet
{'question': 'Show employees with above-average salaries'}


In [ ]:
#%pip install flask flask-cors

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


from flask import Flask
from flask import request
from flask import jsonify

from flask_cors import CORS

app = Flask(__name__)

CORS(app)

print("Flask ready")

@app.route(
    "/ask",
    methods=["POST"]
)
def ask():

    data = request.json

    question = data["question"]

    result = run_pipeline(
        question
    )

    return jsonify(result)

app.run(
    host="0.0.0.0",
    port=5000
)

In [ ]:
#%pip install gradio

INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/20.1 MB 1.6 MB/s eta 0:00:13
   ---------------------------------------- 0.1/20.1 MB 1.6 MB/s eta 0:00:13
   ---------------------------------------- 0.1/20.1 MB 1.6 MB/s eta 0:00:13
   ---------------------------------------- 0.2/20.1 MB 1.2 MB/s eta 0:00:17
   ---------------------------------------- 0.2/20.1 MB 1.2 MB/s eta 0:00:17
   ---------------------------------------- 0.2/20.1 MB 1.2 MB/s eta 0:00:17
   ---------------------------------------- 0.2/20.1 MB 1.2 MB/s eta 0:00:17
   ---------------------------------------- 0.2/20.1 MB 1.2 MB/s eta 0:00:17
   ---------------------------------------- 0.2/20.1 MB 1.2 MB/s eta 0:00:17
   --------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aider-chat 0.86.2 requires fastapi==0.128.8, but you have fastapi 0.136.3 which is incompatible.
aider-chat 0.86.2 requires hf-xet==1.2.0, but you have hf-xet 1.5.0 which is incompatible.
aider-chat 0.86.2 requires huggingface-hub==1.4.1, but you have huggingface-hub 1.15.0 which is incompatible.
aider-chat 0.86.2 requires psutil==7.2.2, but you have psutil 5.9.8 which is incompatible.
aider-chat 0.86.2 requires rich==14.3.2, but you have rich 13.9.4 which is incompatible.
aider-chat 0.86.2 requires starlette==0.52.1, but you have starlette 1.2.1 which is incompatible.
aider-chat 0.86.2 requires tiktoken==0.12.0, but you have tiktoken 0.7.0 which is incompatible.
aider-chat 0.86.2 requires typer==0.23.0, but you have typer 0.25.1 which is incompatible.
aider-chat 0.86.2 requires typer-slim==0.23.0, but you have ty

In [16]:
import gradio as gr

In [17]:
def demo_chat(question):

    return {
        "question": question,
        "answer": "Pipeline not connected yet"
    }

In [18]:
demo = gr.Interface(
    fn=demo_chat,
    inputs="text",
    outputs="json",
    title="AI Database Assistant"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
